# Tech Challenge 2 - Interpretacao de Resultados com LLM

Este notebook atende ao bloco de integracao com LLMs: utiliza uma LLM pre-treinada via Groq para transformar a classificacao do modelo em uma explicacao textual destinada a revisao medica.

A LLM nao realiza novo diagnostico. Ela recebe a probabilidade prevista, as cinco principais contribuicoes locais derivadas do modelo servido e insights estruturados calculados a partir dessas evidencias, sem identificadores do paciente.

Na Fase 2, Regressao Logistica, KNN e Arvore de Decisao foram otimizados e comparados. Este notebook interpreta o modelo final selecionado para serving em `modelo_serving.joblib`; no experimento atual, esse modelo e a Regressao Logistica. Portanto, a interpretacao operacional explica o diagnostico que a API efetivamente entregaria, nao uma comparacao simultanea entre os tres modelos.


## Estrategia de prompt engineering

O prompt separa instrucoes fixas, contexto numerico e insights estruturados. As instrucoes obrigam:

- linguagem em portugues orientada a profissional de saude;
- uso do termo `resultado do modelo`, sem afirmar diagnostico confirmado;
- quatro secoes padronizadas: resumo, evidencias, insights acionaveis e limitacoes;
- transformacao dos sinais numericos em pontos de revisao medica;
- proibicao de prescricao terapeutica;
- declaracao da natureza academica do modelo e da ausencia de validacao externa.

A versao do prompt e `clinical_explanation_v2`, com modelo configuravel por variavel de ambiente.


In [ ]:
from pathlib import Path
import json
import os
import sys

import joblib
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluate_llm import evaluate_cases, predict_case, select_representative_cases
from src.llm_interpretation import (
    DEFAULT_LLM_MODEL,
    PROMPT_VERSION,
    SYSTEM_INSTRUCTIONS,
    build_interpretation_prompt,
    derive_actionable_insights,
    derive_feature_evidence,
)

MODEL_PATH = ROOT / 'resultados' / 'fase2' / 'modelo_serving.joblib'
DATA_PATH = ROOT / 'data' / 'cancer_mama.csv'
OUTPUT_DIR = ROOT / 'resultados' / 'fase2'
artifact = joblib.load(MODEL_PATH)

_llm = os.getenv('LLM_MODEL', DEFAULT_LLM_MODEL)
print(f"Modelo interpretado: {artifact['model_label']}")
print(f"LLM configurada: {_llm}")
print(f"Versao do prompt: {PROMPT_VERSION}")
print(f"GROQ_API_KEY configurada: {bool(os.getenv('GROQ_API_KEY'))}")


## Casos representativos para avaliacao

A avaliacao seleciona perfis deterministicos: maior probabilidade maligna, maior probabilidade benigna, caso mais proximo do limiar de decisao e amostras adicionais por faixas de probabilidade. Assim, o texto e testado em cenarios de alto risco, baixo risco, incerteza relativa e perfis intermediarios.


In [2]:
casos = select_representative_cases(artifact, DATA_PATH)
resumo_casos = []
for nome, row in casos:
    resultado = predict_case(artifact, row)
    resumo_casos.append({
        'caso': nome,
        'classificacao_estimada': resultado.diagnosis,
        'probabilidade_maligna': resultado.probability_malignant,
        'probabilidade_benigna': resultado.probability_benign,
    })
pd.DataFrame(resumo_casos).style.format({
    'probabilidade_maligna': '{:.2%}',
    'probabilidade_benigna': '{:.2%}',
})

,caso,classificacao_estimada,probabilidade_maligna,probabilidade_benigna
0,alto_risco_maligno,Maligno,100.00%,0.00%
1,alto_risco_benigno,Benigno,0.00%,100.00%
2,caso_limiar,Maligno,54.60%,45.40%


## Contexto enviado a LLM

Abaixo esta o contexto do caso de maior probabilidade maligna. Somente dados derivados necessarios a explicacao sao enviados a LLM.

In [ ]:
nome_caso, row = casos[0]
resultado = predict_case(artifact, row)
evidencias = derive_feature_evidence(artifact, row.to_dict())
insights = derive_actionable_insights(resultado, evidencias)

display(pd.DataFrame([item.__dict__ for item in evidencias]))
display(pd.DataFrame([item.__dict__ for item in insights]))
print(build_interpretation_prompt(resultado, evidencias))


## Avaliacao da qualidade das interpretacoes

Quando `GROQ_API_KEY` estiver configurada, a proxima celula gera explicacoes reais para os casos selecionados e salva:

- `resultados/fase2/interpretacoes_llm.json`, com textos, metadados, evidencias e insights acionaveis;
- `resultados/fase2/avaliacao_interpretacoes_llm.csv`, com a rubrica objetiva por caso;
- `resultados/fase2/resumo_avaliacao_llm.json`, com score medio, minimo, maximo e criterios reprovados.

A rubrica verifica: mencao da classe prevista, probabilidade, secoes obrigatorias, insights acionaveis, limitacao explicita, orientacao de revisao profissional e ausencia de prescricao. A avaliacao automatica deve ser complementada por revisao humana de um profissional de saude.


In [ ]:
if os.getenv('GROQ_API_KEY'):
    avaliacao = evaluate_cases(MODEL_PATH, DATA_PATH, OUTPUT_DIR)
    display(avaliacao.style.format({'probabilidade_maligna': '{:.2%}', 'score_objetivo': '{:.2%}'}))
    resumo_path = OUTPUT_DIR / 'resumo_avaliacao_llm.json'
    if resumo_path.exists():
        resumo = json.loads(resumo_path.read_text(encoding='utf-8'))
        display(pd.DataFrame([resumo]))
else:
    print('GROQ_API_KEY nao configurada.')
    print('Configure a variavel de ambiente antes de reexecutar a avaliacao real da LLM.')


## Uso pela API

Com `GROQ_API_KEY` definida e `python src/api.py` em execucao, o endpoint `POST /interpret` recebe as mesmas 30 features do `/predict` e retorna classificacao, probabilidades, evidencias locais, insights acionaveis estruturados, explicacao da LLM, versao do prompt e verificacoes da rubrica.

O campo `model` da resposta corresponde ao pipeline unico carregado pela API. Como o artefato de serving atual foi escolhido apos a comparacao dos tres modelos, a resposta retorna `Regressao Logistica` por decisao de publicacao do melhor modelo observado.

O endpoint `/metrics` registra quantidade de interpretacoes, latencia e distribuicao da nota objetiva, permitindo acompanhar a camada LLM em operacao.
